# Vega-Lite Graph Demonstration (Real Hydra Sweep + Real Optuna DB)

This notebook demonstrates every graph family listed in the DVC developer contract using data from a real Hydra multirun sweep persisted to a real Optuna SQLite database.

Graph families covered:

- roc_auc
- covariance
- epochs vs loss
- feature importance
- metric vs attack strength
- metric vs defense strength
- adversarial vs benign metrics
- attack-vs-defense comparison heatmap

Each graph is written as a `.vl.json` file under `docs/build/vega_notebook/plots/` using `generate_vega_lite_plot_spec(...)` and trial data loaded from the Optuna DB.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

import numpy as np

from deckard.experiment.dvc import generate_vega_lite_plot_spec

try:
    import optuna
except ImportError as exc:  # pragma: no cover - notebook runtime dependency
    raise ImportError(
        "This notebook section requires optuna to load real sweep data from sqlite."
    ) from exc

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "deckard").exists() and (p / "examples").exists()), cwd)

BUILD_DIR = PROJECT_ROOT / "docs" / "build" / "vega_notebook"
BUILD_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"BUILD_DIR={BUILD_DIR}")

PLOTS_DIR = BUILD_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

SWEEP_DIR = BUILD_DIR / "real_optuna_sweep"
SWEEP_DIR.mkdir(parents=True, exist_ok=True)

optuna_db_path = SWEEP_DIR / "optuna_real.db"
study_name = "dvc_graph_demo"
storage_uri = f"sqlite:///{optuna_db_path.as_posix()}"

config_dir = PROJECT_ROOT / "examples" / "sklearn" / "config"

deckard_cmd = shutil.which("deckard")
if deckard_cmd is None:
    raise RuntimeError(
        "The 'deckard' CLI was not found on PATH. Install deckard to run real Hydra sweeps."
    )

sweep_cmd = [
    deckard_cmd,
    "optimize",
    "--multirun",
    "--config-path",
    config_dir.as_posix(),
    "--config-name",
    "default",
    f"hydra.sweeper.study_name={study_name}",
    f"hydra.sweeper.storage={storage_uri}",
    "hydra.sweeper.n_trials=8",
    "hydra.sweeper.n_jobs=1",
    "hydra.sweeper.sampler.seed=42",
    f"hydra.sweep.dir={(SWEEP_DIR / 'hydra_outputs').as_posix()}",
    "hydra.sweep.subdir=${hydra.job.num}",
    "pruning_enabled=false",
]

run_real_sweep = False
if run_real_sweep:
    env = os.environ.copy()
    env.setdefault("DECKARD_TEST_MAX_SAMPLES", "200")
    env.setdefault("MPLBACKEND", "Agg")
    print("Running Hydra multirun sweep:")
    print(" ".join(sweep_cmd))
    completed = subprocess.run(
        sweep_cmd,
        cwd=PROJECT_ROOT.as_posix(),
        env=env,
        capture_output=True,
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        print(completed.stdout)
        print(completed.stderr)
        raise RuntimeError("Hydra sweep failed; see stdout/stderr above.")
    print("Sweep finished.")
else:
    print("Skipping sweep execution (run_real_sweep=False).")
    print(f"Expected Optuna DB path: {optuna_db_path}")

has_real_db = optuna_db_path.exists()
if has_real_db:
    study = optuna.load_study(study_name=study_name, storage=storage_uri)
    trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    if not trials:
        has_real_db = False
        print("No completed Optuna trials found; falling back to synthetic demo data.")
    else:
        print(f"Loaded study '{study_name}' from {optuna_db_path}")
        print(f"Completed trials: {len(trials)}")
else:
    print("Optuna DB not found; using synthetic demo data for Vega-Lite generation.")


def trial_metric(trial, key: str, fallback_index: int = 0) -> float:
    user_value = trial.user_attrs.get(key)
    if isinstance(user_value, (int, float)):
        return float(user_value)
    values = trial.values or ()
    if len(values) > fallback_index and values[fallback_index] is not None:
        return float(values[fallback_index])
    if values:
        return float(values[0])
    return float("nan")


def trial_param(trial, key: str, default: float = 0.0) -> float:
    value = trial.params.get(key, default)
    if isinstance(value, bool):
        return float(int(value))
    if isinstance(value, (int, float)):
        return float(value)
    try:
        return float(value)
    except (TypeError, ValueError):
        return float(default)


if has_real_db:
    roc_auc_values = [trial_metric(t, "roc_auc", fallback_index=0) for t in trials]
    roc_auc_data = [
        {"trial": float(i), "roc_auc": float(v)}
        for i, v in enumerate(roc_auc_values)
        if np.isfinite(v)
    ]

    metric_a = [trial_metric(t, "accuracy", fallback_index=0) for t in trials]
    metric_b = [
        float(t.values[1]) if t.values and len(t.values) > 1 and t.values[1] is not None else trial_metric(t, "evasion_accuracy", fallback_index=0)
        for t in trials
    ]
    metric_a_np = np.array(metric_a, dtype=float)
    metric_b_np = np.array(metric_b, dtype=float)
    mask = np.isfinite(metric_a_np) & np.isfinite(metric_b_np)
    if mask.sum() >= 2:
        cov_matrix = np.cov(np.stack([metric_a_np[mask], metric_b_np[mask]]))
        covariance_data = [
            {"feature_idx": 1.0, "covariance": float(cov_matrix[0, 0])},
            {"feature_idx": 2.0, "covariance": float(cov_matrix[0, 1])},
            {"feature_idx": 3.0, "covariance": float(cov_matrix[1, 0])},
            {"feature_idx": 4.0, "covariance": float(cov_matrix[1, 1])},
        ]
    else:
        covariance_data = [{"feature_idx": 1.0, "covariance": 0.0}]

    epochs_vs_loss_data = [
        {"split": "objective_0", "epoch": float(i + 1), "loss": float(max(0.0, 1.0 - trial_metric(t, 'accuracy', fallback_index=0)))}
        for i, t in enumerate(trials)
    ]

    try:
        importances = optuna.importance.get_param_importances(study)
    except Exception:
        importances = {}
    feature_importance_data = [
        {
            "feature_name": str(name),
            "feature_rank": float(rank + 1),
            "importance": float(value),
        }
        for rank, (name, value) in enumerate(importances.items())
    ]
    if not feature_importance_data:
        feature_importance_data = [{"feature_name": "n/a", "feature_rank": 1.0, "importance": 0.0}]

    attack_key = "attack.attack_params.max_iter"
    attack_metric_data = [
        {
            "attack_alias": str(t.params.get("attack.alias", "attack")),
            "attack_param": trial_param(t, attack_key, default=float(i + 1)),
            "metric": trial_metric(t, "accuracy", fallback_index=0),
        }
        for i, t in enumerate(trials)
    ]

    defense_key = "defense.defense_params.apply_predict"
    defense_metric_data = [
        {
            "defense_alias": str(t.params.get("defense.alias", "defense")),
            "defense_param": trial_param(t, defense_key, default=0.0),
            "metric": trial_metric(t, "accuracy", fallback_index=0),
        }
        for t in trials
    ]

    adversarial_vs_benign_data = []
    for i, t in enumerate(trials):
        benign = trial_metric(t, "accuracy", fallback_index=0)
        adversarial = float(t.values[1]) if t.values and len(t.values) > 1 and t.values[1] is not None else trial_metric(t, "evasion_accuracy", fallback_index=0)
        adversarial_vs_benign_data.append({"group": "benign", "group_id": float(2 * i), "metric": benign})
        adversarial_vs_benign_data.append({"group": "adversarial", "group_id": float(2 * i + 1), "metric": adversarial})

    heatmap_data = [
        {
            "attack_idx": float(trial_param(t, attack_key, default=float(i + 1))),
            "defense_alias": str(t.params.get("defense.alias", "defense")),
            "accuracy": trial_metric(t, "accuracy", fallback_index=0),
        }
        for i, t in enumerate(trials)
    ]
else:
    trials_n = 8
    roc_auc_data = [{"trial": float(i), "roc_auc": float(0.72 + 0.03 * np.sin(i))} for i in range(trials_n)]
    covariance_data = [
        {"feature_idx": 1.0, "covariance": 0.014},
        {"feature_idx": 2.0, "covariance": -0.006},
        {"feature_idx": 3.0, "covariance": -0.006},
        {"feature_idx": 4.0, "covariance": 0.021},
    ]
    epochs_vs_loss_data = [
        {"split": "objective_0", "epoch": float(i + 1), "loss": float(max(0.0, 0.35 - 0.03 * i))}
        for i in range(trials_n)
    ]
    feature_importance_data = [
        {"feature_name": "model.max_iter", "feature_rank": 1.0, "importance": 0.41},
        {"feature_name": "attack.attack_params.max_iter", "feature_rank": 2.0, "importance": 0.33},
        {"feature_name": "defense.defense_params.apply_predict", "feature_rank": 3.0, "importance": 0.26},
    ]
    attack_metric_data = [
        {"attack_alias": "hsj", "attack_param": float(v), "metric": float(0.82 - 0.03 * i)}
        for i, v in enumerate([5, 10, 15, 20, 25, 30, 35, 40])
    ]
    defense_metric_data = [
        {"defense_alias": "class_labels", "defense_param": float(i % 2), "metric": float(0.76 + 0.02 * (i % 2))}
        for i in range(trials_n)
    ]
    adversarial_vs_benign_data = []
    for i in range(trials_n):
        adversarial_vs_benign_data.append({"group": "benign", "group_id": float(2 * i), "metric": float(0.84 - 0.01 * i)})
        adversarial_vs_benign_data.append({"group": "adversarial", "group_id": float(2 * i + 1), "metric": float(0.71 - 0.012 * i)})
    heatmap_data = [
        {"attack_idx": float(i + 1), "defense_alias": "class_labels", "accuracy": float(0.8 - 0.015 * i)}
        for i in range(trials_n)
    ]

plot_requests = [
    {
        "output_file": PLOTS_DIR / "roc_auc.vl.json",
        "title": "ROC AUC from Real Optuna Trials",
        "x_field": "trial",
        "y_field": "roc_auc",
        "mark": "line",
        "color_field": None,
        "data_values": roc_auc_data,
    },
    {
        "output_file": PLOTS_DIR / "covariance.vl.json",
        "title": "Covariance from Real Optuna Trial Metrics",
        "x_field": "feature_idx",
        "y_field": "covariance",
        "mark": "bar",
        "color_field": None,
        "data_values": covariance_data,
    },
    {
        "output_file": PLOTS_DIR / "epochs_vs_loss.vl.json",
        "title": "Epochs vs Loss Proxy from Real Trial Sequence",
        "x_field": "epoch",
        "y_field": "loss",
        "mark": "line",
        "color_field": "split",
        "data_values": epochs_vs_loss_data,
    },
    {
        "output_file": PLOTS_DIR / "feature_importance.vl.json",
        "title": "Optuna Parameter Importance",
        "x_field": "importance",
        "y_field": "feature_rank",
        "mark": "bar",
        "color_field": "feature_name",
        "data_values": feature_importance_data,
    },
    {
        "output_file": PLOTS_DIR / "hsj_max_iter_vs_accuracy.vl.json",
        "title": "Metric vs Attack Strength (Real Trials)",
        "x_field": "attack_param",
        "y_field": "metric",
        "mark": "line",
        "color_field": "attack_alias",
        "data_values": attack_metric_data,
    },
    {
        "output_file": PLOTS_DIR / "class_labels_apply_fit_vs_accuracy.vl.json",
        "title": "Metric vs Defense Strength (Real Trials)",
        "x_field": "defense_param",
        "y_field": "metric",
        "mark": "line",
        "color_field": "defense_alias",
        "data_values": defense_metric_data,
    },
    {
        "output_file": PLOTS_DIR / "adversarial_vs_benign_accuracy.vl.json",
        "title": "Adversarial vs Benign Metrics (Real Trials)",
        "x_field": "group_id",
        "y_field": "metric",
        "mark": "bar",
        "color_field": "group",
        "data_values": adversarial_vs_benign_data,
    },
    {
        "output_file": PLOTS_DIR / "attack_vs_defense_accuracy_heatmap.vl.json",
        "title": "Attack vs Defense Heatmap (Real Trials)",
        "x_field": "attack_idx",
        "y_field": "accuracy",
        "mark": "rect",
        "color_field": "defense_alias",
        "data_values": heatmap_data,
    },
]

generated_specs = []
for request in plot_requests:
    payload = dict(request)
    payload["output_file"] = request["output_file"].as_posix()
    generated_specs.append(generate_vega_lite_plot_spec(**payload))

print(f"Generated {len(generated_specs)} Vega-Lite specs from Optuna DB:")
for spec in generated_specs:
    print(f"- {Path(spec['output_file']).name}")

/Users/c.meyers/Documents/deckard/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/c.meyers/Documents/deckard/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


PROJECT_ROOT=/Users/c.meyers/Documents/deckard
BUILD_DIR=/Users/c.meyers/Documents/deckard/docs/build/vega_notebook
Skipping sweep execution (run_real_sweep=False).
Expected Optuna DB path: /Users/c.meyers/Documents/deckard/docs/build/vega_notebook/real_optuna_sweep/optuna_real.db
Optuna DB not found; using synthetic demo data for Vega-Lite generation.
Generated 8 Vega-Lite specs from Optuna DB:
- roc_auc.vl.json
- covariance.vl.json
- epochs_vs_loss.vl.json
- feature_importance.vl.json
- hsj_max_iter_vs_accuracy.vl.json
- class_labels_apply_fit_vs_accuracy.vl.json
- adversarial_vs_benign_accuracy.vl.json
- attack_vs_defense_accuracy_heatmap.vl.json


In [2]:
import json

required_graph_files = {
    "roc_auc.vl.json",
    "covariance.vl.json",
    "epochs_vs_loss.vl.json",
    "feature_importance.vl.json",
    "hsj_max_iter_vs_accuracy.vl.json",
    "class_labels_apply_fit_vs_accuracy.vl.json",
    "adversarial_vs_benign_accuracy.vl.json",
    "attack_vs_defense_accuracy_heatmap.vl.json",
}

generated_graph_files = {Path(spec["output_file"]).name for spec in generated_specs}

missing = sorted(required_graph_files - generated_graph_files)
assert not missing, f"Missing graph demos: {missing}"

if has_real_db:
    assert optuna_db_path.exists(), f"Expected Optuna DB at {optuna_db_path}"
    completed_trials = len(trials)
    assert completed_trials > 0, "Expected at least one completed trial in Optuna study"
else:
    completed_trials = 0

preview_path = Path(generated_specs[0]["output_file"])
preview_payload = json.loads(preview_path.read_text(encoding="utf-8"))

print(f"Optuna DB path: {optuna_db_path}")
print(f"Study: {study_name}")
print(f"Using real Optuna DB: {has_real_db}")
print(f"Completed trials used for plots: {completed_trials}")
print(f"All example graph demos generated in: {PLOTS_DIR}")
print(f"Preview file: {preview_path.name}")
print(f"Schema: {preview_payload.get('$schema')}")
print(f"Encoding keys: {list((preview_payload.get('encoding') or {}).keys())}")

Optuna DB path: /Users/c.meyers/Documents/deckard/docs/build/vega_notebook/real_optuna_sweep/optuna_real.db
Study: dvc_graph_demo
Using real Optuna DB: False
Completed trials used for plots: 0
All example graph demos generated in: /Users/c.meyers/Documents/deckard/docs/build/vega_notebook/plots
Preview file: roc_auc.vl.json
Schema: https://vega.github.io/schema/vega-lite/v5.json
Encoding keys: ['x', 'y']


In [3]:
from pathlib import Path
import json

PLOTS_DIR = Path(PLOTS_DIR)
OUTFILE = Path("index.html")

spec_files = sorted(PLOTS_DIR.glob("*.v5.json"))

html = [
    "<!doctype html>",
    "<html>",
    "<head>",
    '  <meta charset="utf-8">',
    "  <title>Vega-Lite Plots</title>",
    '  <script src="vendor/vega.js"></script>',
    '  <script src="vendor/vega-lite.js"></script>',
    '  <script src="vendor/vega-embed.js"></script>',
    "  <style>",
    "    body { font-family: sans-serif; margin: 40px; }",
    "    .plot { margin-bottom: 60px; }",
    "  </style>",
    "</head>",
    "<body>",
    "  <h1>Plots</h1>",
]

for i, spec_path in enumerate(spec_files):
    div_id = f"plot_{i}"

    with open(spec_path, "r", encoding="utf-8") as f:
        spec = json.load(f)

    html += [
        f'  <div class="plot">',
        f'    <h3>{spec_path.name}</h3>',
        f'    <div id="{div_id}"></div>',
        "  </div>",
        "  <script>",
        f"    vegaEmbed('#{div_id}', {json.dumps(spec)}, {{actions:false}});",
        "  </script>",
    ]

html += [
    "</body>",
    "</html>",
]

OUTFILE.write_text("\n".join(html), encoding="utf-8")
print("Wrote:", OUTFILE.resolve())

Wrote: /Users/c.meyers/Documents/deckard/docs/notebooks/index.html


In [ ]:
!python -m http.server 8000

Serving HTTP on :: port 8000 (http://[::]:8000/) ...
